In [2]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error
from scipy.optimize import minimize
import matplotlib.pyplot as plt
import json

plt.style.use('fivethirtyeight')
%matplotlib inline

In [3]:
# read in data
data = pd.read_csv("tqc_clean.csv")
data = data[['months', 'act0', 'act1', 'CPUE', 'biomass']]
data.head()

,months,act0,act1,CPUE,biomass
0,3,-0.911334,-0.843671,-1.0,-1.0
1,4,-0.964028,-0.703425,-1.0,-1.0
2,5,-0.964028,-0.904936,-1.0,-1.0
3,6,-0.884734,-0.851039,-1.0,-1.0
4,7,-0.839049,-0.761919,-1.0,-1.0


In [ ]:
# remove anomolous biomass data
data = data[(data['biomass'] != -1) & (data['biomass'] <= -0.2)]

In [5]:
# objective function for MSE
def objective_function(var, space, num_bins = None, bin_widths = None):
    curr_data = data[['act0', 'act1', 'CPUE', 'biomass']].copy()

    if num_bins:
        bin_mapper = lambda x: bin_map_creator(x, space, num_bins, None)
    else:
        bin_mapper = lambda x: bin_map_creator(x, space, None, bin_widths)

    bin_midpoint = lambda x: (x[1] + x[0]) / 2

    curr_data['bin'] = curr_data[var].apply(bin_mapper)
    curr_data['midpoint'] = curr_data['bin'].apply(bin_midpoint)
    return mean_squared_error(curr_data[var], curr_data['midpoint'])

In [6]:
# function that maps continuous value to bin
def bin_map_creator(val, space, num_bins = None, bin_widths = None):
    if num_bins:
        bin_boundaries = np.linspace(space[0], space[1], num_bins + 1)
    else:
        bin_boundaries = np.array([])
        total = -1
        for width in bin_widths:
            bin_boundaries = np.append(bin_boundaries, total)
            total += width
        bin_boundaries = np.append(bin_boundaries, total)
    bins = np.array([(bin_boundaries[i], bin_boundaries[i+1]) for i in range(len(bin_boundaries) - 1)])
    
    for curr_bin in bins:
        left, right = curr_bin
        if left <= val < right:
            return curr_bin
    return bins[-1]

In [7]:
# search for optimal bin width to discretize continuous variables using numerical optimization
optimal_width_data_all = {}

# get ranges for variables
action_space = (-1, 1)
cpue_space = (-1, np.max(data['CPUE']))
biomass_space = (-1, np.max(data['biomass']))

data_vars = ['act0', 'act1', 'CPUE', 'biomass']
spaces = [action_space, action_space, cpue_space, biomass_space]
bins = [6, 6, 6, 4]

In [8]:
# function to run in scipy.minimize

def minimize_var_by_width(var, space, num_bins):

    # set up width objective functions
    # mean squared error between the original continuous values and the bin midpoints
    width_objective_func = lambda x: objective_function(var, space, None, x)
    space_length = space[1] - space[0]

    # create constraint for optimizer: bin widths add up to total space
    width_constraints = {"type": "eq", "fun": lambda widths: np.sum(widths) - space_length}

    # set width bounds
    width_bounds = np.array([(1e-3, space_length) for i in np.arange(num_bins)])

    # start from evenly spaced points
    initial_guess = np.random.dirichlet(np.ones(num_bins)) * space_length

    # run sequential least squares programming (constrained continuous optimization)
    optimal_width_data = minimize(width_objective_func, initial_guess, method='SLSQP', bounds=width_bounds, constraints=width_constraints)
    
    # store 
    if var not in optimal_width_data_all.keys():
        optimal_width_data_all[var] = []
    optimal_width_data_all[var].append(optimal_width_data)
    #print(f"Current Variable: {var}\nNumber of Bins: {num_bins}\nMonth: {month_dict[month]}\n{optimal_width_data}\n")

In [9]:
# run optimization for all variables
for i in range(len(data_vars)):
    var = data_vars[i]
    space = spaces[i]
    minimize_var_by_width(var = var, space = space, num_bins = bins[i])

In [10]:
# convert optimization results to bin boundary pairs for each variable
optimal_bins_all = {}

for i in range(len(data_vars)):
    var = data_vars[i]
    space = spaces[i]
        
    optimal_widths = np.round(optimal_width_data_all[var][0].x, 3)
        
    optimal_bin_boundaries = np.array([])
    total = space[0]
    for width in optimal_widths:
        optimal_bin_boundaries = np.append(optimal_bin_boundaries, total)
        total += width
    optimal_bin_boundaries = np.append(optimal_bin_boundaries, total)
        
    optimal_bins = np.array([(optimal_bin_boundaries[i], optimal_bin_boundaries[i+1]) for i in range(len(optimal_bin_boundaries) - 1)])
    optimal_bins_all[var] = optimal_bins

In [11]:
# save as json
optimal_bins_serializable = {
    f"{var}": bounds.tolist()
    for var, bounds in optimal_bins_all.items()
}

with open("optimal_bins.json", "w") as f:
    json.dump(optimal_bins_serializable, f)